In [ ]:
import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import Polygon, MultiPolygon

In [ ]:
LABEL = "lubuskie"
OSM_ID = 130969
AOI = f"lubuskie_130969.geojson"
MAP_HEX_SIZE = 7

In [ ]:
def get_hexes_for_polygon(polygon, target_hex_size=7, fill_hex_size=10):
    polygon_points_lon_lat = polygon.exterior.coords
    polygon_points_lat_lon = tuple(coord[::-1] for coord in polygon_points_lon_lat)
    h3_polygon = h3.LatLngPoly(polygon_points_lat_lon)
    fill_hexes = h3.h3shape_to_cells(h3_polygon, fill_hex_size)
    target_hexes = {h3.cell_to_parent(h, target_hex_size) for h in fill_hexes}
    return target_hexes

In [ ]:
gdf_aoi = gpd.read_file(AOI)
gdf_aoi

In [ ]:
geometry = gdf_aoi["geometry"].iloc[0]
geometry

In [ ]:
aoi_hexes = set()
if isinstance(geometry, Polygon):
    aoi_hexes = get_hexes_for_polygon(polygon=geometry, target_hex_size=MAP_HEX_SIZE)
elif isinstance(geometry, MultiPolygon):
    for geom_polygon in geometry.geoms:
        aoi_hexes.update(get_hexes_for_polygon(polygon=geom_polygon, target_hex_size=MAP_HEX_SIZE))

In [ ]:
df_output = pd.DataFrame(data=aoi_hexes, columns=["h3_index"])
df_output["osm_id"] = OSM_ID
df_output["name"] = LABEL
df_output = df_output[["osm_id", "name", "h3_index"]]
df_output.head()

In [ ]:
df_output.to_csv(f"hex_list_{OSM_ID}_h{MAP_HEX_SIZE}.csv", index=False)